# Загрузка данных

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [2]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


# Исследование данных

## Аномалии и пропуски
Рассмотрим основные статистические характеристики для выявления аномалий и пропуском в датасетах.

### Train и test

In [3]:
train.describe(include='all').loc[["count", "unique", "min", "mean", "max"]]

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
count,11091,11091,11091,11091,11091.000000
unique,11091,NaN,NaN,NaN,NaN
min,NaN,2025-01-22 17:09:47,2026-04-06 00:00:00,2026-04-07 00:00:00,0.000000
mean,NaN,2026-01-30 07:44:22.135966208,2026-04-12 04:33:10.370570752,2026-04-13 04:33:10.370571008,0.081057
max,NaN,2026-04-18 22:45:04,2026-04-19 00:00:00,2026-04-20 00:00:00,1.000000


In [4]:
test.describe(include='all').loc[["count", "unique", "min", "mean", "max"]]

,cookie_id,cookie_created_at,window_start_ts,window_end_ts
count,4909,4909,4909,4909
unique,4909,NaN,NaN,NaN
min,NaN,2025-01-15 16:57:05,2026-04-20 00:00:00,2026-04-21 00:00:00
mean,NaN,2026-02-10 12:42:52.963740160,2026-04-23 04:14:37.082908928,2026-04-24 04:14:37.082908928
max,NaN,2026-04-25 23:00:54,2026-04-26 00:00:00,2026-04-27 00:00:00


В тренировочных и тестовых данных никаких пропуском и аномальных значений не наблюдается. Все `cookie_id` уникальны, а `event_ts` имеют корректный формат и тестовый диапазон идет после тренировочного.

Отсюда полезно посчитать возраст куки как разницу между `cookie_created_at` и `window_start`, так как боты - это часто новые пользователи.

### Events

Сразу уберем дубликаты строк из `events`, если такие имеются, так как они могут повлиять на качество модели.

In [5]:
print("Количество дублирующийся строк в events:", events.duplicated().sum())
events = events.drop_duplicates()

Количество дублирующийся строк в events: 4863


In [6]:
events.describe(include='all')

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
count,324042,324042,324042.000000,324042,324042,324042,2.111040e+05,291584,300574,192154,98929,98929.000000,106966.000000,106966.000000
unique,16000,NaN,NaN,10,11,148,NaN,15,40,2,120,NaN,NaN,NaN
top,ck_3461e957e7ed4c93,NaN,NaN,item_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,avtomobili,sankt-peterburg,private,кофемашина,NaN,NaN,NaN
freq,187,NaN,NaN,118990,46157,3688,NaN,20536,38579,110955,907,NaN,NaN,NaN
mean,NaN,2026-04-15 20:42:22.213114368,212.133190,NaN,NaN,NaN,1.029992e+06,NaN,NaN,NaN,NaN,2.902395,913.377101,535.585326
min,NaN,2026-04-06 00:02:57,100.000000,NaN,NaN,NaN,1.000000e+06,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000
25%,NaN,2026-04-10 18:38:16.249999872,100.000000,NaN,NaN,NaN,1.014983e+06,NaN,NaN,NaN,NaN,1.000000,458.000000,282.000000
50%,NaN,2026-04-15 03:39:48.500000,200.000000,NaN,NaN,NaN,1.030038e+06,NaN,NaN,NaN,NaN,2.000000,884.000000,533.000000
75%,NaN,2026-04-20 17:07:22.750000128,210.000000,NaN,NaN,NaN,1.044955e+06,NaN,NaN,NaN,NaN,4.000000,1358.000000,789.000000
max,NaN,2026-04-26 23:59:50,900.000000,NaN,NaN,NaN,1.059999e+06,NaN,NaN,NaN,NaN,65.000000,1920.000000,1080.000000


По статистическим характеристикам видно, что численные и временные значения не имеют выбросов.
У признаком есть много пропуском - рассмотрим их подробнее.

In [7]:
print("Пропуски по колонкам в events:")
events.isna().mean().round(3)

Пропуски по колонкам в events:


cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.349
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64

Пропуски в данных имеют структурную природу. Например, `item_*`, `seller_type` и `search_*`: эти поля появляются только для определённых типов событий.

Поэтому заполнять такие пропуски средним, медианой или предыдущим значением бесполезно и даже вредно: мы создадим данные о действиях, которых на самом деле не было. Например, позиция курсора (`pointer_x`/`pointer_y`) на телефоне. Интерполяция здесь тоже неприменима, потому что соседние события могут относиться к разным типам и не образуют непрерывный ряд.

Вместо восстановления значения можно добавить сам факт отсутствия как информацию: считаем долю пропусков для каждой `cookie_id`, а числовые и категориальные агрегаты считаем только по существующим значениям.

## Анализ признаков
Теперь посмотрим значения каждого из признаков. Это позволит выявить полезные признаки для модели, сгенерировать новые признаки и понять, какие признаки можно удалить из датасета.

### `event_ts`

In [8]:
events["event_ts"].describe()

count                           324042
mean     2026-04-15 20:42:22.213114368
min                2026-04-06 00:02:57
25%      2026-04-10 18:38:16.249999872
50%         2026-04-15 03:39:48.500000
75%      2026-04-20 17:07:22.750000128
max                2026-04-26 23:59:50
Name: event_ts, dtype: object

Время события - это первый самый важный признак, так как боты и люди могут совершать разные сценарии действия. Например, боты могут совершать действия с одинаковым интервалом или высокой скоростью, а люди - с разным и непредсказуемым интервалом.

В сочетании с другими признаками можно узнать добавляет ли пользователь товары в избранное после просмотра, продолжает искать объявление интересующей его категории, совершает ли он покупки в разное время суток и т.д.

### `eid` и `event_name`

In [9]:
events[["eid", "event_name"]].value_counts().reset_index().sort_values("eid")

,eid,event_name,count
1,100,search_results_view,98929
0,200,item_view,118990
2,210,photo_swipe,35962
4,220,seller_page_view,17180
5,300,contact_phone_show,11154
8,301,contact_chat_open,6047
9,303,contact_message_sent,3032
3,400,favorite_add,18739
7,500,login,6178
6,900,captcha_shown,7831


`eid` и `event_name` описывают, что именно делала кука. Это вторые из самых важных источников признаков: боты и люди могут иметь одинаковое количество событий, но совершенно отличные последовательности действий. 

`eid` - это идентификатор (группа) события, а `event_name` - название события. С помощью этих двух полей можно построить признаки, которые будут описывать поведение пользователя. Например, доля событий определенного типа, количество уникальных событий, последовательность событий и т.д.

### `platform`

In [10]:
events["platform"].value_counts()

platform
desktop    46157
WEB        45799
Web        45703
web        45284
ANDROID    41809
Android    41637
android    41542
iOS         4035
iphone      4035
IOS         4028
ios         4013
Name: count, dtype: int64

Колонку `platform` нужно привести к единому виду: нижний регистр, 'iphone' в 'ios'. Сама платформа не доказывает, что пользователь является ботом или человеком, но всё равно может быть полезна. Рассмотрим подробнее разнообразие платформ, которые использует пользователь.

In [11]:
events["platform"] = events["platform"].str.lower().replace({"iphone": "ios"})
events["platform"].value_counts()

platform
web        136786
android    124988
desktop     46157
ios         16111
Name: count, dtype: int64

In [12]:
(pd.crosstab(events["cookie_id"], events["platform"]) > 0).value_counts()

android  desktop  ios    web  
False    True     False  True     7773
True     False    False  False    6586
False    False    True   False     834
                  False  True      759
         True     False  False      48
Name: count, dtype: int64

Как показывает наш анализ, половина пользователей используют только одну платформу и друга — web и desktop, что можно считать тоже за одну платформу. Таким образом, признак малоинформативен, но основную платформу или их разнообразие полезно использовать для более точной классификации. Например, люди чаще пользуется телефоном, а боты могут использовать несколько платформ для маскировки.

### `user_agent`

In [13]:
events["user_agent"].head(10)

0    Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...
1    Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...
2    Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
3    Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...
4    Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...
5    Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...
6     Avito/122.0 (Android 11; M2101K6G) okhttp/4.11.0
7    Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
8    Mozilla/5.0 (iPhone; CPU iPhone OS 15_2 like M...
9     Avito/118.0 (Android 12; SM-A515F) okhttp/4.11.0
Name: user_agent, dtype: object

`user_agent` - это строка, которая содержит информацию о браузере, операционной системе и устройстве пользователя. Она больше содержит технической информации: однозначно можно получить только клиент пользователя (Mozilla/5.0 | Avito/118.0) и ОС (Macintosh | Android). Можно назвать дублированием `platform` с небольшим добавлением информации. Однако, user-agent строки могут быть легко подделаны, поэтому этот признак не является надежным индикатором.

In [14]:
def get_ua_client(ua):
    if pd.isna(ua):
        return np.nan
    
    if ua.startswith("Mozilla"):
        return 'web'
    if ua.startswith("Avito"):
        return 'avito_app'

    return "other"

In [15]:
events["ua_client"] = events["user_agent"].apply(get_ua_client)
events["ua_client"].value_counts()

ua_client
web          257620
avito_app     59442
other          6980
Name: count, dtype: int64

In [16]:
events.groupby("cookie_id")["ua_client"].nunique().value_counts()

ua_client
1    16000
Name: count, dtype: int64

In [17]:
events[events["ua_client"] == "other"]["user_agent"].value_counts()

user_agent
node-fetch/3.3.2                       1275
curl/8.1.2                             1195
Scrapy/2.11.0 (+https://scrapy.org)    1190
python-urllib3/2.0.7                   1180
Go-http-client/1.1                     1179
python-requests/2.31.0                  961
Name: count, dtype: int64

In [18]:
pd.crosstab(events["ua_client"], events["platform"])[["ios", "android", "desktop", "web"]]

platform,ios,android,desktop,web
ua_client,,,,
avito_app,10898,48544,0,0
other,0,0,1773,5207
web,5213,76444,44384,131579


In [19]:
def get_ua_platform(ua):
    if pd.isna(ua):
        return np.nan
    
    ua = ua.lower()
    if 'windows' in ua:
        return 'windows'
    elif 'macintosh' in ua:
        return 'mac'
    elif 'android' in ua:
            return 'android'
    elif 'linux' in ua:
        return 'linux'
    elif 'iphone' in ua or 'ipad' in ua:
        return 'ios'
    else:
        return 'other'

In [20]:
events["ua_platform"] = events["user_agent"].apply(get_ua_platform)
events.groupby("cookie_id")["ua_platform"].nunique().value_counts()

ua_platform
1    15349
2      651
Name: count, dtype: int64

In [21]:
pd.crosstab(events["platform"], events["ua_platform"]).T

platform,android,desktop,ios,web
ua_platform,,,,
android,124988,0,0,0
ios,0,0,16111,0
linux,0,3094,0,9176
mac,0,10347,0,31365
other,0,1773,0,5207
windows,0,30943,0,91038


In [22]:
(pd.crosstab(events["cookie_id"], events["ua_platform"]) > 0).value_counts()

android  ios    linux  mac    other  windows
True     False  False  False  False  False      6586
False    False  False  False  False  True       5476
                       True   False  False      1760
         True   False  False  False  False       834
         False  False  True   False  True        651
                True   False  False  False       421
                False  False  True   False       272
Name: count, dtype: int64

В данных между `user_agent` и `platform` никаких аномальных значений нет. Эти признаки дублируют между собой значения, но модели будет важно отличать меняет ли пользователь устройства или использует приложение Avito.

### `item_id`

In [23]:
print("Количество уникальных item_id:", events["item_id"].nunique(dropna=False))
events["item_id"].value_counts(dropna=False).head(10)

Количество уникальных item_id: 53857


item_id
NaN          112938
1046399.0        32
1036649.0        31
1012082.0        31
1006132.0        30
1058692.0        29
1050950.0        29
1014070.0        28
1032850.0        28
1008950.0        28
Name: count, dtype: int64

Идентификатор объявления содержит большое количество уникальных значений, что делает его малоинформативным для модели. Однако, можно использовать его для создания признаков, таких как количество уникальных объявлений, с которыми взаимодействовал пользователь.

### `item_category` и `item_location`

In [24]:
events["item_category"].value_counts(dropna=False)

item_category
NaN                  32458
avtomobili           20536
mebel                20236
bytovaya_tehnika     19861
hobbi                19731
elektronika          19711
uslugi               19586
odezhda              19559
kvartiry_arenda      19429
kvartiry_prodazha    19404
noutbuki             19277
detskie_tovary       19139
telefony             19099
rabota               19084
zhivotnye            18823
zapchasti            18109
Name: count, dtype: int64

In [25]:
events.groupby("cookie_id")["item_category"].nunique().value_counts()

item_category
2    5478
1    4710
3    3598
4    1296
5     507
6     239
7     122
0      26
8      24
Name: count, dtype: int64

In [26]:
events.groupby("cookie_id")["item_category"].nunique(dropna=False).value_counts()

item_category
3    4601
2    4576
4    2812
1    2080
5    1111
6     455
7     225
8     116
9      24
Name: count, dtype: int64

In [27]:
print("Количество уникальных item_location:", events["item_location"].nunique(dropna=False))
events["item_location"].value_counts(dropna=False).head(10)

Количество уникальных item_location: 41


item_location
sankt-peterburg    38579
novosibirsk        37447
moskva             37386
ekaterinburg       36732
NaN                23468
ryazan              4564
izhevsk             4522
barnaul             4479
astrahan            4453
ufa                 4417
Name: count, dtype: int64

In [28]:
events.groupby("cookie_id")["item_location"].nunique().value_counts()

item_location
2     1982
3     1900
4     1752
5     1498
6     1303
1     1262
7     1029
8      862
9      752
10     569
11     549
12     453
13     358
14     329
15     282
16     205
17     174
18     160
19     113
21      82
20      81
23      63
22      62
24      47
25      32
26      27
27      22
0       16
28      15
30       9
29       6
31       4
32       2
Name: count, dtype: int64

Категория и локация товара описывают группы объявлений, с которыми взаимодействовал пользователь. С помощью них можно отследить поведение пользователя: ищет ли он объявления в одной категории или в разных, в одном регионе или в разных. Например, боты могут искать объявления в разных категориях и регионах, чтобы увеличить охват.

Сочетая с `event_ts` и `event_name`, можно полные сценарии поведения пользователя, которые будут очень информативными для модели.

### `seller_type`

In [29]:
events['seller_type'].value_counts(dropna=False)

seller_type
NaN        131888
private    110955
pro         81199
Name: count, dtype: int64

Тип продавца имеет всего 2 уникальных значения (не считая пропусков). Признак никак не влияет на поведение пользователя, поэтому его можно не использовать в дальнейшем.

### `search_query`

In [30]:
print("Количество уникальных search_query:", events["search_query"].nunique(dropna=False))
events["search_query"].value_counts(dropna=False).head(10)

Количество уникальных search_query: 121


search_query
NaN                     225113
кофемашина                 907
toyota camry               906
renault logan              900
kia rio 2019               897
клининг                    896
посудомоечная машина       891
телевизор 55               888
вторичка 2к                888
шкаф купе                  887
Name: count, dtype: int64

Боты часто строят одинаковые поисковые запросы между собственными сессиями. Можно использовать признак количества уникальных запросов и вместе с `item_category` изучить сценарии взаимодействия: не уходит ли человек с категории товаров и ищет до покупки или добавлении в избранное.

### `search_page`

In [31]:
events["search_page"].describe()

count    98929.000000
mean         2.902395
std          2.885921
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         65.000000
Name: search_page, dtype: float64

### `pointer_x` и `pointer_y`

In [32]:
events[["pointer_x", "pointer_y"]].describe()

,pointer_x,pointer_y
count,106966.000000,106966.000000
mean,913.377101,535.585326
std,538.416149,303.929918
min,0.000000,0.000000
25%,458.000000,282.000000
50%,884.000000,533.000000
75%,1358.000000,789.000000
max,1920.000000,1080.000000


По изменениям координат курсора можно понять пользователь относиться к боту или человеку. Человек ведет между событиями мышку плавно не по прямой линии в отличие от бота.

# Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [33]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

events_train = events_in_window(events, train)
events_test = events_in_window(events, test)

# Бейзлайн

## Агрегация признаков

Каждая строка `train` и `test` соответствует одной `cookie_id`, поэтому события сначала агрегируются до уровня куки. Для обучения модель получает одну строку признаков на одну куку.

Все события уже ограничены суточным окном наблюдения. Это важно: события за его пределами нельзя использовать, иначе появится утечка из будущего.

### Группы признаков и обоснование

- **Масштаб активности:** количество событий, уникальных типов событий и `eid`. Бот может генерировать существенно больше однотипных действий.
- **Возраст куки:** разница между `cookie_created_at` и началом окна. Свежие куки могут быть связаны с автоматическим трафиком.
- **Разнообразие объектов:** число уникальных объявлений, категорий, локаций, продавцов и поисковых запросов. Это показывает ширину интересов и охват сбора данных.
- **Состав действий:** доли `event_name`, `platform`, `ua_client` и `ua_platform`. Они описывают сценарий и техническую среду, в которой происходят события.
- **Временная активность:** длительность активности, число уникальных временных точек, число активных часов и максимальное число событий за час. Это отражает интенсивность и распределение активности.
- **Интервалы:** среднее, медиана, разброс, квантили, мода и доли коротких интервалов. Автоматические запросы часто имеют регулярный или слишком быстрый темп.
- **Повторы:** доля одинаковых событий подряд. Длинные повторяющиеся серии могут указывать на массовый автоматический просмотр.
- **Числовые поля:** статистики `search_page` и координат курсора. Они считаются только там, где поле имеет смысл; искусственное заполнение исходных пропусков не выполняется.
- **Движение курсора:** длина шага между соседними координатами. У человека и автоматического клиента могут отличаться характер и разнообразие движений.

Общие доли пропусков не добавляются: они в основном объясняются типом события и дублируют доли `event_name`. Сами пропуски сохраняются только на этапе расчёта агрегатов: значения считаются по существующим данным, а технические `NaN` после объединения заменяются на `0`.

In [34]:
def agg_features(events_df, meta):
    features = meta[["cookie_id"]].copy()
    grouped = events_df.groupby("cookie_id", sort=False)

    # Считаем масштаб и разнообразие активности: одна строка результата = одна cookie.
    basic = pd.DataFrame({
        "n_events": grouped.size(),
        "n_unique_event_types": grouped["event_name"].nunique(),
        "n_unique_eids": grouped["eid"].nunique(),
    })
    features = features.merge(basic, left_on="cookie_id", right_index=True, how="left")

    # Возраст cookie доступен до конца окна и не использует будущие события.
    meta_time = meta[["cookie_id", "cookie_created_at", "window_start_ts"]].copy()
    meta_time["cookie_age_days"] = (
        meta_time["window_start_ts"] - meta_time["cookie_created_at"]
    ).dt.total_seconds() / 86400
    features = features.merge(
        meta_time[["cookie_id", "cookie_age_days"]],
        on="cookie_id",
        how="left",
    )

    # Считаем разнообразие объектов только по существующим значениям.
    # Общие missing_share убраны: они в основном дублируют event_name_share_*.
    for column in [
        "item_id",
        "item_category",
        "item_location",
        "seller_type",
        "search_query",
    ]:
        features[f"{column}_nunique"] = grouped[column].nunique().reindex(
            features["cookie_id"]
        ).to_numpy()

    # Доли категорий описывают сценарий cookie независимо от общего числа событий.
    def add_shares(column, prefix=None):
        share_table = pd.crosstab(
            events_df["cookie_id"],
            events_df[column],
            normalize="index",
        )
        prefix = prefix or column
        share_table.columns = [f"{prefix}_share_{value}" for value in share_table.columns]
        return share_table

    for column in ["event_name", "platform", "ua_client", "ua_platform"]:
        shares = add_shares(column)
        features = features.merge(
            shares,
            left_on="cookie_id",
            right_index=True,
            how="left",
        )

    # Сортировка событий нужна, чтобы интервалы и последовательности считались по времени.
    timestamps = (
        events_df[["cookie_id", "event_ts", "event_name"]]
        .dropna(subset=["event_ts"])
        .sort_values(["cookie_id", "event_ts"])
        .copy()
    )
    timestamps["interval_seconds"] = (
        timestamps.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    )

    # Длительность активности и число уникальных временных точек характеризуют поток событий.
    activity = timestamps.groupby("cookie_id").agg(
        first_event_ts=("event_ts", "min"),
        last_event_ts=("event_ts", "max"),
        n_unique_event_timestamps=("event_ts", "nunique"),
    )
    activity["active_duration_seconds"] = (
        activity["last_event_ts"] - activity["first_event_ts"]
    ).dt.total_seconds()
    activity = activity.drop(columns=["first_event_ts", "last_event_ts"])
    features = features.merge(
        activity,
        left_on="cookie_id",
        right_index=True,
        how="left",
    )

    # Статистики интервалов помогают отличать регулярные и слишком быстрые действия ботов.
    intervals = timestamps.dropna(subset=["interval_seconds"])

    def mode_or_zero(values):
        modes = values.mode()
        return modes.iloc[0] if not modes.empty else 0.0

    interval_features = intervals.groupby("cookie_id")["interval_seconds"].agg(
        n_intervals="count",
        interval_mean_seconds="mean",
        interval_median_seconds="median",
        interval_std_seconds="std",
        interval_min_seconds="min",
        interval_max_seconds="max",
        interval_q10_seconds=lambda values: values.quantile(0.10),
        interval_q25_seconds=lambda values: values.quantile(0.25),
        interval_q75_seconds=lambda values: values.quantile(0.75),
        interval_q90_seconds=lambda values: values.quantile(0.90),
        interval_mode_seconds=mode_or_zero,
        interval_short_share=lambda values: (values <= 10).mean(),
        interval_minute_share=lambda values: (values <= 60).mean(),
        interval_hour_share=lambda values: (values <= 3600).mean(),
    )
    interval_features["interval_cv"] = interval_features["interval_std_seconds"] / (
        interval_features["interval_mean_seconds"] + 1e-3
    )
    features = features.merge(
        interval_features,
        left_on="cookie_id",
        right_index=True,
        how="left",
    )

    # Распределение активности по часам показывает непрерывность и концентрацию сессии.
    timestamps["hour"] = timestamps["event_ts"].dt.hour
    timestamps["hour_group"] = pd.cut(
        timestamps["hour"],
        bins=[-1, 5, 11, 17, 23],
        labels=["night", "morning", "afternoon", "evening"],
    )
    hour_counts = timestamps.groupby(["cookie_id", "hour"], observed=False).size()
    features["active_hours_nunique"] = timestamps.groupby("cookie_id")["hour"].nunique().reindex(
        features["cookie_id"]
    ).to_numpy()
    features["max_events_per_hour"] = hour_counts.groupby(level=0).max().reindex(
        features["cookie_id"]
    ).to_numpy()
    for hour_group in ["night", "morning", "afternoon", "evening"]:
        group_share = timestamps.groupby("cookie_id")["hour_group"].apply(
            lambda values, group=hour_group: (values == group).mean()
        )
        features[f"hour_share_{hour_group}"] = group_share.reindex(
            features["cookie_id"]
        ).to_numpy()

    # Доля одинаковых соседних событий отражает повторяющиеся автоматические сценарии.
    event_sequence = timestamps.copy()
    event_sequence["previous_event"] = event_sequence.groupby("cookie_id")["event_name"].shift()
    event_sequence["is_repeat"] = (
        event_sequence["event_name"] == event_sequence["previous_event"]
    )
    repeat_share = event_sequence.groupby("cookie_id")["is_repeat"].mean()
    features["consecutive_same_event_share"] = repeat_share.reindex(
        features["cookie_id"]
    ).to_numpy()

    # Для числовых полей считаем статистики только у событий, где поле применимо.
    for column in ["search_page", "pointer_x", "pointer_y"]:
        if column in events_df:
            numeric_values = events_df[["cookie_id", column]].copy()
            numeric_values[column] = pd.to_numeric(numeric_values[column], errors="coerce")
            numeric_stats = numeric_values.groupby("cookie_id")[column].agg(
                mean="mean",
                std="std",
                min="min",
                max="max",
                median="median",
            )
            numeric_stats.columns = [f"{column}_{name}" for name in numeric_stats.columns]
            features = features.merge(
                numeric_stats,
                left_on="cookie_id",
                right_index=True,
                how="left",
            )

    # Изменение координат курсора даёт поведенческий сигнал для desktop/web-событий.
    points = events_df[["cookie_id", "event_ts", "pointer_x", "pointer_y"]].copy()
    points[["pointer_x", "pointer_y"]] = points[["pointer_x", "pointer_y"]].apply(
        pd.to_numeric, errors="coerce"
    )
    points = points.sort_values(["cookie_id", "event_ts"])
    points["dx"] = points.groupby("cookie_id")["pointer_x"].diff()
    points["dy"] = points.groupby("cookie_id")["pointer_y"].diff()
    points["pointer_step"] = np.sqrt(points["dx"] ** 2 + points["dy"] ** 2)
    pointer_features = points.groupby("cookie_id")["pointer_step"].agg(
        pointer_move_mean="mean",
        pointer_move_median="median",
        pointer_move_max="max",
        pointer_move_std="std",
    )
    features = features.merge(
        pointer_features,
        left_on="cookie_id",
        right_index=True,
        how="left",
    )

    # Скорость активности: защищаем знаменатель от деления на ноль.
    features["events_per_active_hour"] = features["n_events"] / (
        features["active_duration_seconds"].div(3600).clip(lower=1 / 3600)
    )

    # Остаточные NaN появляются у cookie без подходящих событий; для модели заменяем их на 0.
    features = features.replace([np.inf, -np.inf], np.nan).fillna(0)
    return features

In [35]:
X_train = agg_features(events_train, train).set_index("cookie_id")
X_test = agg_features(events_test, test).set_index("cookie_id")
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train = train.set_index("cookie_id")["target"]

X_train.head()

,n_events,n_unique_event_types,n_unique_eids,cookie_age_days,item_id_nunique,item_category_nunique,item_location_nunique,seller_type_nunique,search_query_nunique,event_name_share_contact_chat_open,...,pointer_y_mean,pointer_y_std,pointer_y_min,pointer_y_max,pointer_y_median,pointer_move_mean,pointer_move_median,pointer_move_max,pointer_move_std,events_per_active_hour
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,3,3,135.603692,4,3,5,2,3,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,150.898204
ck_7e4de46eeab82974,41,7,7,194.576111,19,1,12,2,5,0.000000,...,567.605263,272.953569,104.0,1053.0,601.0,680.385017,660.474479,1762.325736,381.072559,3.738981
ck_9320229ef6304522,36,9,9,32.994421,19,7,9,2,10,0.027778,...,556.657143,312.027426,22.0,1057.0,560.0,967.287900,843.048041,1790.046089,424.201490,3.551561
ck_30ccd25bc1714ed9,21,5,5,0.546759,10,1,4,2,3,0.047619,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,4.389989
ck_a77c5f05948cdeef,28,6,6,94.837095,16,4,6,2,6,0.000000,...,639.740741,295.966031,5.0,1041.0,692.0,621.438369,597.850316,1239.108147,279.732374,5.278040


## Деление данных

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

In [36]:
test_start_date = test["window_start_ts"].min().normalize()
print("Самая ранняя дата в test:", test_start_date)


train_dates = pd.Series(train["window_start_ts"].dt.normalize().unique()).sort_values()
candidate_split_dates = train_dates[train_dates < test_start_date].tail(8).tolist()

print("Даты-кандидаты для начала X_val:")
print(*candidate_split_dates, sep="\n")

Самая ранняя дата в test: 2026-04-20 00:00:00
Даты-кандидаты для начала X_val:
2026-04-12 00:00:00
2026-04-13 00:00:00
2026-04-14 00:00:00
2026-04-15 00:00:00
2026-04-16 00:00:00
2026-04-17 00:00:00
2026-04-18 00:00:00
2026-04-19 00:00:00


In [37]:
split_report = []
y_values = y_train.to_numpy()
for candidate_date in candidate_split_dates:
    candidate_valid = (train["window_start_ts"].dt.normalize() >= candidate_date).to_numpy()
    split_report.append({
        "split_date": candidate_date,
        "train_size": int((~candidate_valid).sum()),
        "val_size": int(candidate_valid.sum()),
        "val_share": round(float(candidate_valid.mean()), 4),
        "val_target_1_share": round(float(y_values[candidate_valid].mean()), 4),
    })
print(pd.DataFrame(split_report).to_string(index=False))

split_date  train_size  val_size  val_share  val_target_1_share
2026-04-12        5113      5978     0.5390              0.0795
2026-04-13        5930      5161     0.4653              0.0791
2026-04-14        6773      4318     0.3893              0.0822
2026-04-15        7599      3492     0.3148              0.0816
2026-04-16        8400      2691     0.2426              0.0818
2026-04-17        9140      1951     0.1759              0.0820
2026-04-18        9853      1238     0.1116              0.0759
2026-04-19       10459       632     0.0570              0.0665


In [38]:
split_date = pd.Timestamp("2026-04-17")  # выбор даты разбиения

is_valid = (train["window_start_ts"].dt.normalize() >= split_date).to_numpy()
X_fit = X_train.loc[~is_valid]
y_fit = y_train.iloc[~is_valid]
X_val = X_train.loc[is_valid]
y_val = y_train.iloc[is_valid]

print("Выбранная граница:", split_date.date())
print("train объектов:", len(X_fit))
print("val объектов:", len(X_val))
print("test объектов:", len(test))
print("train баланс классов:", y_fit.value_counts(normalize=True).sort_index().round(4).to_dict())
print("val баланс классов:", y_val.value_counts(normalize=True).sort_index().round(4).to_dict())

Выбранная граница: 2026-04-17
train объектов: 9140
val объектов: 1951
test объектов: 4909
train баланс классов: {0: 0.9191, 1: 0.0809}
val баланс классов: {0: 0.918, 1: 0.082}


## Обучение модели

In [39]:
from catboost import CatBoostClassifier
from metric import precision_at_recall

catboost_balanced = CatBoostClassifier(
    iterations=700,
    depth=6,
    learning_rate=0.04,
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_balanced.fit(X_fit, y_fit)
p_val = catboost_balanced.predict_proba(X_val)[:, 1]

print('CatBoost Balanced P@R0.7:', round(
    precision_at_recall(y_val, p_val), 4
))
model = catboost_balanced

CatBoost Balanced P@R0.7: 0.7619


# Улучшения

## Подбор гиперпараметров

In [40]:
import optuna
from catboost import CatBoostClassifier
from metric import precision_at_recall
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 1200),
        "depth": trial.suggest_int("depth", 4, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 32, 254),
    }

    candidate = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="Logloss",
        auto_class_weights="Balanced",
        random_seed=42,
        thread_count=1,
        verbose=False,
        allow_writing_files=False,
    )
    candidate.fit(X_fit, y_fit)
    validation_score = candidate.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, validation_score)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=30, show_progress_bar=False)

print("Лучший P@R0.7:", round(study.best_value, 4))
print("Лучшие параметры:")
print(study.best_params)

catboost_optuna = CatBoostClassifier(
    **study.best_params,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    thread_count=1,
    verbose=False,
    allow_writing_files=False,
)
catboost_optuna.fit(X_fit, y_fit)
p_val_optuna = catboost_optuna.predict_proba(X_val)[:, 1]
print("Проверка лучшей модели P@R0.7:", round(
    precision_at_recall(y_val, p_val_optuna), 4
))
model = catboost_optuna

Лучший P@R0.7: 0.7945
Лучшие параметры:
{'iterations': 644, 'depth': 8, 'learning_rate': 0.08981956638229543, 'l2_leaf_reg': 24.92598441698659, 'random_strength': 1.2328725917611425, 'bagging_temperature': 3.501752619842477, 'border_count': 194}
Проверка лучшей модели P@R0.7: 0.7733


In [41]:
import optuna
from metric import precision_at_recall
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
scale_pos_weight = float((y_fit == 0).sum() / (y_fit == 1).sum())


def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
    }
    candidate = XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        n_jobs=1,
        random_state=42,
    )
    candidate.fit(X_fit, y_fit)
    validation_score = candidate.predict_proba(X_val)[:, 1]
    return precision_at_recall(y_val, validation_score)


xgb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
xgb_study.optimize(xgb_objective, n_trials=30, show_progress_bar=False)

print("XGBoost лучший P@R0.7:", round(xgb_study.best_value, 4))
print("XGBoost лучшие параметры:")
print(xgb_study.best_params)

xgb_model = XGBClassifier(
    **xgb_study.best_params,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    n_jobs=1,
    random_state=42,
)
xgb_model.fit(X_fit, y_fit)
p_val_xgb = xgb_model.predict_proba(X_val)[:, 1]
print("XGBoost проверка P@R0.7:", round(
    precision_at_recall(y_val, p_val_xgb), 4
))

XGBoost лучший P@R0.7: 0.8175
XGBoost лучшие параметры:
{'n_estimators': 473, 'max_depth': 3, 'learning_rate': 0.06714977359272588, 'min_child_weight': 1.4428576396957156, 'subsample': 0.8331990510306982, 'colsample_bytree': 0.9062923939092716, 'gamma': 2.714006218904295, 'reg_alpha': 9.357784839631398, 'reg_lambda': 0.032209671553132105}
XGBoost проверка P@R0.7: 0.8175


In [42]:
from sklearn.metrics import average_precision_score

In [43]:
import optuna
from lightgbm import LGBMClassifier
from metric import precision_at_recall

optuna.logging.set_verbosity(optuna.logging.WARNING)


def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 150),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
    }
    candidate = LGBMClassifier(
        **params,
        objective="binary",
        class_weight="balanced",
        verbosity=-1,
        n_jobs=1,
        random_state=42,
    )
    candidate.fit(X_fit, y_fit)
    validation_score = candidate.predict_proba(X_val)[:, 1]
    return precision_at_recall(y_val, validation_score)


lgbm_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
lgbm_study.optimize(lgbm_objective, n_trials=30, show_progress_bar=False)

print("LightGBM лучший P@R0.7:", round(lgbm_study.best_value, 4))
print("LightGBM лучшие параметры:")
print(lgbm_study.best_params)

lgbm_model = LGBMClassifier(
    **lgbm_study.best_params,
    objective="binary",
    class_weight="balanced",
    verbosity=-1,
    n_jobs=1,
    random_state=42,
)
lgbm_model.fit(X_fit, y_fit)
p_val_lgbm = lgbm_model.predict_proba(X_val)[:, 1]
print("LightGBM проверка P@R0.7:", round(
    precision_at_recall(y_val, p_val_lgbm), 4
))

LightGBM лучший P@R0.7: 0.8116
LightGBM лучшие параметры:
{'n_estimators': 937, 'num_leaves': 19, 'max_depth': 12, 'learning_rate': 0.09528587217040241, 'min_child_samples': 39, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 0.003320559103751959, 'reg_lambda': 0.22356209648036393}
LightGBM проверка P@R0.7: 0.8116


In [44]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

p_val_average = (p_val_optuna + p_val_xgb + p_val_lgbm) / 3
print('Среднее трех моделей P@R0.7:', round(
    precision_at_recall(y_val, p_val_average), 4
))

stack_estimators = [
    (
        'catboost',
        CatBoostClassifier(
            **study.best_params,
            loss_function='Logloss',
            eval_metric='AUC',
            auto_class_weights='Balanced',
            random_seed=42,
            thread_count=1,
            verbose=False,
            allow_writing_files=False,
        ),
    ),
    (
        'xgboost',
        XGBClassifier(
            **xgb_study.best_params,
            objective='binary:logistic',
            eval_metric='auc',
            scale_pos_weight=scale_pos_weight,
            tree_method='hist',
            n_jobs=1,
            random_state=42,
        ),
    ),
    (
        'lightgbm',
        LGBMClassifier(
            **lgbm_study.best_params,
            objective='binary',
            class_weight='balanced',
            verbosity=-1,
            n_jobs=1,
            random_state=42,
        ),
    ),
]

stacking_model = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(
            penalty='l2',
            class_weight='balanced',
            max_iter=1000,
            random_state=42,
        ),
    ),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=1,
)
stacking_model.fit(X_fit, y_fit)
p_val_stacking = stacking_model.predict_proba(X_val)[:, 1]

print('StackingClassifier P@R0.7:', round(
    precision_at_recall(y_val, p_val_stacking), 4
))
model = stacking_model

Среднее трех моделей P@R0.7: 0.7887
StackingClassifier P@R0.7: 0.7887


/Users/potemeyker/Projects/bot_detection_challenge/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/potemeyker/Projects/bot_detection_challenge/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/potemeyker/Projects/bot_detection_challenge/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/potemeyker/Projects/bot_detection_challenge/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/potemeyker/Projects/bot_detection_challenge/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered

Лучший скор с подбором гиперпараметров показала XGBoost. Остальные методы оказались немного слабее.

Из-за не хватки времени не получилось сравнить модели и методы с помощью бустрапа, что позволило построить доверительный интервалы качества и выбрать более устойчивую и точную модель.

## Добавление новых признаков

Из-за нехватки времени не удалось реализовать модульный пайплайн генерации признаков. Идея заключалась в том, чтобы разбить монолитную функцию расчета на небольшие независимые функции по смысловым блокам: время, поиск, действия с контактами и поведение мыши. Каждая функция принимает события с метаданными и возвращает датафрейм признаков по cookie_id, после чего результаты просто соединяются через pd.concat. Такой подход позволил бы быстро отключать и подключать группы признаков, чтобы наглядно замерить вклад каждой из них в метрику P@R >= 0.7.

С точки зрения самих признаков основной упор планировался на специфику скраперов. В поиске это анализ пагинации: боты уходят намного глубже первой страницы и листают выдачу строго последовательно (1, 2, 3 и далее). Для сбора контактов сильным сигналом стало бы отношение кликов по телефону к числу открытых объявлений, так как у обычных людей это редкое действие. В поведении курсора главным маркером автоматизации является полное отсутствие координат мыши на десктопе, что выдает запуск через headless-браузеры вроде Selenium. Наконец, во времени помог бы расчет энтропии интервалов и поиск фиксированных пауз вроде ровно одной секунды между запросами, характерных для скриптов.

# Сабмит

In [45]:
best_model = LGBMClassifier(
    **lgbm_study.best_params,
    objective="binary",
    class_weight="balanced",
    verbosity=-1,
    n_jobs=1,
    random_state=42,
)

In [46]:
best_model.fit(X_train, y_train)
sub = pd.DataFrame({
    'cookie_id': X_test.index,
    'score': best_model.predict_proba(X_test)[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,1.924551e-07
1,ck_a76ee3b3e3e522fd,7.919881e-03
2,ck_94c9a4d382689e82,1.048943e-03
3,ck_8eaf9509ad9462a0,5.765354e-08
4,ck_9a88a5a989cb5bc6,6.234385e-04
